# RingVoz — Segmentación por clustering (modelo descriptivo no supervisado)

**Integrantes:** José Becerra · Marleny Guiral Marín · Geraldine Suárez  
**Responsable de este notebook:** Geraldine Suárez

Modelo descriptivo no supervisado que agrupa las transacciones por similitud sin usar `is_fraud` como entrada. Cubre la fila *descriptivo / no supervisado* de la tabla de modelos analíticos del entregable CRISP-DM y complementa al modelo predictivo (`modelado_riesgo_ringvoz.ipynb`).

- Entrada: `dataset_fraude_ringvoz_class_last.arff` (8 predictoras; `is_fraud` excluida del input).
- Métodos: K-Means (k = 2..10) y clustering jerárquico aglomerativo (Ward).
- Métricas de validez interna: silueta, Davies–Bouldin, Calinski–Harabasz.

In [ ]:
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE_PATH = '/content/'
ARFF_PATH = BASE_PATH + 'dataset_fraude_ringvoz_class_last.arff'
OUT_CSV   = BASE_PATH + 'clusters_kmeans.csv'
OUT_JSON  = BASE_PATH + 'resultados_clustering.json'

In [ ]:
def read_arff(path):
    cols, types, data_lines = [], {}, []
    in_data = False
    with open(path, 'r', encoding='utf-8') as f:
        for raw in f:
            s = raw.strip()
            if not s or s.startswith('%'):
                continue
            if s.upper().startswith('@DATA'):
                in_data = True; continue
            if in_data:
                data_lines.append(s)
            elif s.upper().startswith('@ATTRIBUTE'):
                m = re.match(r'@attribute\s+"?([\w_]+)"?\s+(.*)', s, re.IGNORECASE)
                if m:
                    cols.append(m.group(1))
                    types[m.group(1)] = 'numeric' if m.group(2).strip().upper().startswith('NUMERIC') else 'nominal'
    pat = re.compile(r"'([^']*)'|([^,]+)")
    rows = [[m.group(1) if m.group(1) is not None else m.group(2).strip()
             for m in pat.finditer(l)] for l in data_lines]
    df = pd.DataFrame([r for r in rows if len(r) == len(cols)], columns=cols)
    for c, t in types.items():
        if t == 'numeric':
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

df = read_arff(ARFF_PATH)
print(f'Filas: {len(df):,} | Columnas: {list(df.columns)}')
print(f'is_fraud: {df["is_fraud"].value_counts().to_dict()}')

## Preprocesamiento

`is_fraud` se excluye del input (clustering no supervisado) y se guarda para el cross-tab posterior. Numéricas estandarizadas y categóricas con one-hot por requisito de distancia euclídea de K-Means.

In [ ]:
y_label = df['is_fraud'].astype(str)
X = df.drop(columns=['is_fraud'])
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

pre = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
])
X_proc = pre.fit_transform(X)
print('Espacio de features:', X_proc.shape)

## Búsqueda del k óptimo (K-Means, k = 2..10)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
idx_sample = rng.choice(X_proc.shape[0], size=10_000, replace=False)
X_sample = X_proc[idx_sample]

filas = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_proc)
    labels = km.labels_
    filas.append({
        'k': k,
        'inercia': km.inertia_,
        'silueta': silhouette_score(X_sample, labels[idx_sample]),
        'davies_bouldin': davies_bouldin_score(X_proc, labels),
        'calinski_harabasz': calinski_harabasz_score(X_proc, labels),
    })
metricas = pd.DataFrame(filas)
metricas

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(metricas['k'], metricas['inercia'], 'o-');            ax[0].set_title('Codo (inercia)');           ax[0].set_xlabel('k')
ax[1].plot(metricas['k'], metricas['silueta'], 'o-', color='g'); ax[1].set_title('Silueta (mayor = mejor)');  ax[1].set_xlabel('k')
ax[2].plot(metricas['k'], metricas['davies_bouldin'], 'o-', color='r'); ax[2].set_title('Davies–Bouldin (menor = mejor)'); ax[2].set_xlabel('k')
plt.tight_layout(); plt.show()

**Resultado de la búsqueda.** La silueta es máxima en **k = 2 (silueta = 0,2541)** y cae para k ≥ 3. Davies–Bouldin también marca su mínimo en k = 2 (1,2733). Los dos criterios coinciden: la estructura natural del dataset son **dos grupos**; particiones más finas fragmentan segmentos coherentes y bajan la calidad.

In [ ]:
K_OPT = int(metricas.loc[metricas['silueta'].idxmax(), 'k'])
kmeans = KMeans(n_clusters=K_OPT, n_init=20, random_state=RANDOM_STATE).fit(X_proc)
df['cluster_kmeans'] = kmeans.labels_
print(f'k óptimo (silueta máxima) = {K_OPT}')
print('Tamaño de los clusters:'); print(df['cluster_kmeans'].value_counts().sort_index())

**Tamaños obtenidos.** Cluster 0 ≈ **73.683 transacciones (82 %)** y Cluster 1 ≈ **16.207 (18 %)**. Es un partido desigual pero no degenerado — los dos grupos retienen volumen suficiente para análisis.

## Clustering jerárquico (Ward) — validación independiente

Se aplica sobre la misma muestra de 10.000 transacciones usada para la silueta. El objetivo es verificar que la estructura de dos grupos no depende del algoritmo.

In [ ]:
Z = linkage(X_sample, method='ward')
plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90., show_contracted=True)
plt.title('Dendrograma — Ward linkage (muestra n=10.000)')
plt.xlabel('Cluster'); plt.ylabel('Distancia'); plt.tight_layout(); plt.show()

labels_h = fcluster(Z, t=K_OPT, criterion='maxclust')
sil_h = silhouette_score(X_sample, labels_h)
db_h  = davies_bouldin_score(X_sample, labels_h)
print(f'Jerárquico (k={K_OPT}): silueta = {sil_h:.4f} | Davies-Bouldin = {db_h:.4f}')

**Resultado del contraste.** Jerárquico Ward sobre la muestra: silueta = **0,2518** y Davies–Bouldin = **1,2655**. Coincide casi exactamente con K-Means (0,2541 y 1,2733). El dendrograma muestra una separación dominante en el nivel superior consistente con la partición k=2. **Conclusión metodológica:** los dos grupos son una propiedad del dataset, no un artefacto de K-Means.

## Perfil de cada cluster

In [ ]:
perfil_num = df.groupby('cluster_kmeans')[num_cols].agg(['mean', 'median']).round(2)
perfil_cat = df.groupby('cluster_kmeans')[cat_cols].agg(lambda s: s.mode().iat[0])
display(perfil_num)
display(perfil_cat)

**Interpretación de los perfiles.**

| Variable | Cluster 0 (mayoritario) | Cluster 1 (minoritario) |
|---|---|---|
| Tamaño | 73.683 (82 %) | 16.207 (18 %) |
| `Transaction_Type` modal | Recharge from APM | Merchant Recharge |
| `Merchant` modal | Authorize.Net | **Wallet** |
| `pmntMethodId` media | 1,27 (crédito/débito) | **0,00 (sin tarjeta)** |
| `Amount` media | 27,43 USD | 23,20 USD |
| `hour_of_day` media | 14,0 h | 13,5 h |
| `day_of_week` modal | Friday | Monday |

La separación dominante **no es por monto ni por hora**, sino por **medio de pago**: el Cluster 1 agrupa las transacciones tipo *Merchant Recharge* pagadas con **Wallet** (saldo interno), donde `pmntMethodId` es 0 porque no interviene tarjeta. El Cluster 0 contiene el resto de la operación — recargas vía APM/IVR/App pagadas con tarjeta procesada por Authorize.Net.

## Cross-tab cluster × `is_fraud` (post-hoc)

`is_fraud` no entró al modelo. Se revisa a posteriori cómo se distribuyen los positivos entre clusters.

In [ ]:
ct = pd.crosstab(df['cluster_kmeans'], y_label, margins=True, margins_name='Total')
ct['tasa_fraude'] = (ct['1'] / ct['Total']).round(6)
ct

**Hallazgo principal (descriptivo).** Los **17 casos positivos de `is_fraud` están concentrados en el Cluster 0** (tasa = 0,000231). El **Cluster 1 (Wallet) tiene 0 fraudes** (tasa = 0,0000). Es coherente con la naturaleza del fraude estudiado: las reglas que definen `is_fraud` (`respreasoncode ∈ {27, 35, 65}` — AVS Fraud, lista negra, exceso de intentos) **solo aplican a transacciones con tarjeta**, no a pagos por saldo interno. 

Lectura cuidadosa: con solo 17 positivos no hay potencia estadística para concluir asociación significativa; lo que sí se puede afirmar es que el clustering **separó automáticamente la población de riesgo (transacciones con tarjeta) de la población sin riesgo de este tipo de fraude (Wallet)**. Operativamente: el monitoreo antifraude por reglas de gateway solo necesita aplicarse al Cluster 0.

In [ ]:
df[['cluster_kmeans']].to_csv(OUT_CSV, index=False)
resumen = {
    'k_optimo': K_OPT,
    'metricas_por_k': metricas.to_dict(orient='records'),
    'jerarquico_silueta': float(sil_h),
    'jerarquico_davies_bouldin': float(db_h),
    'tam_clusters': df['cluster_kmeans'].value_counts().sort_index().to_dict(),
}
with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print('Guardado:', OUT_CSV); print('Guardado:', OUT_JSON)

## Conclusiones

- **k óptimo = 2** por silueta (0,2541) y Davies–Bouldin (1,2733). Calinski–Harabasz también máxima en k = 2 (20.863).
- **Validación cruzada con jerárquico Ward**: silueta = 0,2518 y Davies–Bouldin = 1,2655. Los dos algoritmos coinciden → la estructura es propia del dataset.
- **Variable que define la partición**: el medio de pago. Cluster 1 = pagos con **Wallet** (saldo interno, sin tarjeta, *Merchant Recharge*); Cluster 0 = pagos con **tarjeta** procesados por Authorize.Net, en todos los demás canales.
- **Calidad moderada** (silueta ≈ 0,25): los grupos son distinguibles pero no compactos. Es el resultado esperado en un dataset operativo donde hay variabilidad continua dentro de cada grupo.
- **Relación con fraude**: los 17 positivos quedan **todos en Cluster 0**; Cluster 1 (Wallet) no aporta ningún caso porque las reglas técnicas que definen `is_fraud` solo se aplican a transacciones con tarjeta. Hallazgo descriptivo, no inferencial.
- **Uso de negocio**: el modelo permite limitar el monitoreo antifraude por reglas de gateway al Cluster 0, evitando reglas redundantes sobre el Cluster 1 que por diseño no puede disparar esos códigos. La detección caso a caso la resuelve el modelo predictivo supervisado (`modelado_riesgo_ringvoz.ipynb`).
